In [2]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 800

departments = ["Emergency", "Cardiology", "General Medicine", "Neurology", "Orthopedics"]
payers = ["Private", "Medicare", "Medicaid", "Self-Pay"]

df = pd.DataFrame({
    "visit_id": range(1001, 1001 + n),
    "department": np.random.choice(departments, n),
    "payer": np.random.choice(payers, n, p=[0.4, 0.25, 0.2, 0.15]),
    "wait_minutes": np.random.gamma(shape=2.2, scale=14, size=n).round(1),
    "billed": np.random.gamma(shape=3.0, scale=900, size=n).round(2),
})
df["collected"] = (df["billed"] * np.random.uniform(0.35, 1.0, n)).round(2)
df["collection_rate"] = (df["collected"] / df["billed"]).round(3)
df["needs_review"] = ((df["collection_rate"] < 0.5) | (df["wait_minutes"] > 90)).astype(int)

print(df.shape)
print(df.head(10))
print(df["needs_review"].value_counts())
df.to_csv("er_visits.csv", index=False)

# Data quality checks — never load data you haven't inspected
print(df.isnull().sum())                 # any missing values?
print("Duplicate visit_ids:", df["visit_id"].duplicated().sum())
print(df.describe().round(2))            # do the ranges look sane?


import sqlite3

conn = sqlite3.connect("hospital.db")
df.to_sql("visits", conn, if_exists="replace", index=False)

check = pd.read_sql("SELECT COUNT(*) AS total_rows FROM visits", conn)
print(check)


q1 = """
SELECT department,
       COUNT(*) AS visits,
       ROUND(AVG(wait_minutes), 1) AS avg_wait,
       ROUND(AVG(collection_rate), 3) AS avg_collection
FROM visits
GROUP BY department
ORDER BY avg_wait DESC;
"""
print(pd.read_sql(q1, conn))


q2 = """
SELECT payer,
       COUNT(*) AS visits,
       ROUND(AVG(collection_rate), 3) AS avg_collection,
       SUM(needs_review) AS flagged
FROM visits
GROUP BY payer
ORDER BY avg_collection;
"""
print(pd.read_sql(q2, conn))
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

X = df[["wait_minutes", "billed", "collection_rate"]]  # what the model may see
y = df["needs_review"]                                  # what it must predict

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=1)

model = DecisionTreeClassifier(random_state=1)
model.fit(X_train, y_train)

pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))
print(confusion_matrix(y_test, pred))



test = X_test.copy()
test["actual"] = y_test
test["predicted"] = pred

misses = test[test["actual"] != test["predicted"]]
print(misses)

(800, 8)
   visit_id        department  ... collection_rate  needs_review
0      1001         Neurology  ...           0.588             0
1      1002       Orthopedics  ...           0.415             1
2      1003  General Medicine  ...           0.848             0
3      1004       Orthopedics  ...           0.978             0
4      1005       Orthopedics  ...           0.981             0
5      1006        Cardiology  ...           0.912             0
6      1007  General Medicine  ...           0.741             0
7      1008  General Medicine  ...           0.806             1
8      1009  General Medicine  ...           0.535             0
9      1010       Orthopedics  ...           0.760             0

[10 rows x 8 columns]
needs_review
0    615
1    185
Name: count, dtype: int64
visit_id           0
department         0
payer              0
wait_minutes       0
billed             0
collected          0
collection_rate    0
needs_review       0
dtype: int64
Duplicate visit

### Audit of the model's errors
The model scored 99% accuracy (confusion matrix: 151 correct clears,
47 correct flags, 0 false alarms, 2 misses). I audited the 2 misses
instead of accepting the score. Both were visits with wait times of
92.9 and 92.4 minutes, just over the 90-minute threshold, with high
collection rates (0.911 and 0.990). The model learned a cutoff of
roughly 93 minutes from the training data, so both errors are
borderline cases at the boundary, not data or logic problems.
Conclusion: the model reproduces the review rule reliably, and its
only disagreements are explainable.
